In [2]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)
alloy = read('bulk_structure/Mg4Zn7_relaxed.cif')
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 110 atoms, Mg40Zn70


In [3]:
slab = surface(alloy, (1, 1, 0), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need 1.75)")
print(f"Stoichiometric: {'YES' if zn*4 == mg*7 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 220, Mg: 80, Zn: 140
Ratio Zn/Mg: 1.7500 (need 1.75)
Stoichiometric: YES
Symmetric: False


In [4]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < 0.3:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Total atomic layers: 1

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------


In [5]:
z = slab.get_positions()[:, 2]
print(f"z min: {z.min():.3f}")
print(f"z max: {z.max():.3f}")
print(f"z range: {z.max() - z.min():.3f} A")
print(f"Cell z: {slab.cell[2][2]:.3f} A")

# Show z distribution
z_sorted = np.sort(z)
print(f"\nFirst 20 z values:")
for i in range(min(20, len(z_sorted))):
    print(f"  {z_sorted[i]:.3f}")
print(f"\nLast 20 z values:")
for i in range(max(0, len(z_sorted)-20), len(z_sorted)):
    print(f"  {z_sorted[i]:.3f}")

z min: 8.000
z max: 18.346
z range: 10.346 A
Cell z: 26.346 A

First 20 z values:
  8.000
  8.000
  8.227
  8.227
  8.454
  8.454
  8.533
  8.533
  8.658
  8.658
  8.686
  8.686
  8.699
  8.699
  8.745
  8.745
  8.760
  8.760
  8.937
  8.937

Last 20 z values:
  17.555
  17.555
  17.634
  17.634
  17.862
  17.862
  18.088
  18.088
  18.193
  18.193
  18.228
  18.228
  18.269
  18.269
  18.269
  18.269
  18.311
  18.311
  18.346
  18.346


In [6]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers (tol={tol}): {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Total atomic layers (tol=0.02): 92

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Mg2     0.000  |       91        Zn2     0.000 NO <---
      1        Zn2     0.227  |       90        Mg2     0.035 NO <---
      2        Mg2     0.454  |       89        Zn4     0.077 NO <---
      3        Zn2     0.533  |       88        Mg2     0.118 NO <---
      4        Mg2     0.658  |       87        Zn2     0.153 NO <---
      5        Zn4     0.692  |       86        Mg2     0.258 NO <---
      6     Mg2Zn2     0.752  |       85        Zn2     0.485 NO <---
      7        Mg2     0.937  |       84        Mg2     0.712    YES
      8        Zn2     1.173  |       83        Zn2     0.791    YES
      9        Mg2     1.406  |       82        Mg2     0.916    YES
     10        Zn2     1.447  |       81        Zn4     0.950 NO <---
     11        Zn2     1.479  |       80     Mg

In [7]:
print("Searching for removals that give Symmetric=True...\n")

found_any = False
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            is_sym = slab_tr.is_symmetric()
        except:
            is_sym = False
        
        if is_sym:
            sym_tr = np.array(tr.get_chemical_symbols())
            mg_tr = sum(sym_tr == 'Mg')
            zn_tr = sum(sym_tr == 'Zn')
            
            removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
            removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
            
            print(f"  bot_rm={bot_rm}, top_rm={top_rm}: "
                  f"{len(tr)} atoms, Mg{mg_tr} Zn{zn_tr}, "
                  f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
            found_any = True

if not found_any:
    print("No symmetric sub-slab found with up to 7 layers removed from each end")

Searching for removals that give Symmetric=True...

  bot_rm=0, top_rm=5: 208 atoms, Mg76 Zn132, removed 0+12=12
  bot_rm=1, top_rm=6: 204 atoms, Mg72 Zn132, removed 2+14=16
  bot_rm=2, top_rm=7: 200 atoms, Mg72 Zn128, removed 4+16=20


In [8]:
# Remove top 5 layers
removed_top = []
for j in range(n - 5, n):
    removed_top.extend(layer_idx[j])

keep = []
for j in range(0, n - 5):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"Symmetric sub-slab: SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

print(f"\nRemoved top atoms ({len(removed_top)}):")
for j in removed_top:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

rem_mg = sum(sym[j] == 'Mg' for j in removed_top)
rem_zn = sum(sym[j] == 'Zn' for j in removed_top)
print(f"\nRemoved: Mg{rem_mg} Zn{rem_zn} = {len(removed_top)} atoms")

Symmetric sub-slab: SG = P-1
Inversion: f -> [0.40258065 0.04387755 0.99021207] - f

Removed top atoms (12):
  idx=202 Zn (19.916, 12.185, 18.193)
  idx=198 Zn (6.472, 12.185, 18.193)
  idx=142 Mg (27.165, 13.366, 18.228)
  idx=144 Mg (13.721, 13.366, 18.228)
  idx=152 Zn (19.398, 7.601, 18.269)
  idx=154 Zn (5.953, 7.601, 18.269)
  idx=151 Zn (12.675, 7.601, 18.269)
  idx=150 Zn (26.120, 7.601, 18.269)
  idx=143 Mg (25.074, 1.837, 18.311)
  idx=145 Mg (11.630, 1.837, 18.311)
  idx=200 Zn (5.435, 3.017, 18.346)
  idx=196 Zn (18.879, 3.017, 18.346)

Removed: Mg4 Zn8 = 12 atoms


In [9]:
ps_orig = AseAtomsAdaptor().get_structure(slab)

print("Inversion partners of removed atoms:\n")
paired_count = 0
unpaired_count = 0
paired_atoms = []
unpaired_atoms = []

keep_set = set(keep)

for j in removed_top:
    frac = ps_orig[j].frac_coords
    cart = ps_orig.lattice.get_cartesian_coords(frac)
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    dists = np.linalg.norm(slab.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    closest = min(keep_set, key=lambda k: dists[k])
    
    if min_d < 0.5:
        paired_count += 1
        paired_atoms.append(j)
        print(f"  idx={j} {sym[j]} -> paired with idx={closest} ({sym[closest]}) dist={min_d:.3f}")
    else:
        unpaired_count += 1
        unpaired_atoms.append(j)
        print(f"  idx={j} {sym[j]} ({cart[0]:.3f}, {cart[1]:.3f}, {cart[2]:.3f}) -> "
              f"inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f}) UNPAIRED")

print(f"\nPaired: {paired_count}, Unpaired: {unpaired_count}")
unpaired_mg = [j for j in unpaired_atoms if sym[j] == 'Mg']
unpaired_zn = [j for j in unpaired_atoms if sym[j] == 'Zn']
print(f"Unpaired: Mg{len(unpaired_mg)} Zn{len(unpaired_zn)}")

Inversion partners of removed atoms:

  idx=202 Zn (19.916, 12.185, 18.193) -> inv: (20.998, 2.558, 7.895) UNPAIRED
  idx=198 Zn (6.472, 12.185, 18.193) -> inv: (7.553, 2.558, 7.895) UNPAIRED
  idx=142 Mg (27.165, 13.366, 18.228) -> inv: (13.748, 1.377, 7.860) UNPAIRED
  idx=144 Mg (13.721, 13.366, 18.228) -> inv: (0.304, 1.377, 7.860) UNPAIRED
  idx=152 Zn (19.398, 7.601, 18.269) -> inv: (21.516, 7.141, 7.819) UNPAIRED
  idx=154 Zn (5.953, 7.601, 18.269) -> inv: (8.072, 7.141, 7.819) UNPAIRED
  idx=151 Zn (12.675, 7.601, 18.269) -> inv: (28.238, 7.141, 7.819) UNPAIRED
  idx=150 Zn (26.120, 7.601, 18.269) -> inv: (14.794, 7.141, 7.819) UNPAIRED
  idx=143 Mg (25.074, 1.837, 18.311) -> inv: (15.839, 12.906, 7.777) UNPAIRED
  idx=145 Mg (11.630, 1.837, 18.311) -> inv: (29.284, 12.906, 7.777) UNPAIRED
  idx=200 Zn (5.435, 3.017, 18.346) -> inv: (8.590, 11.725, 7.742) UNPAIRED
  idx=196 Zn (18.879, 3.017, 18.346) -> inv: (22.034, 11.725, 7.742) UNPAIRED

Paired: 0, Unpaired: 12
Unpaired: Mg

In [10]:
pairs_keep_move = [
    (202, 198),  # Zn
    (142, 144),  # Mg
    (152, 154),  # Zn
    (151, 150),  # Zn
    (143, 145),  # Mg
    (200, 196),  # Zn
]

sc_fixed = slab.copy()
ps_orig = AseAtomsAdaptor().get_structure(slab)

for keep_idx, move_idx in pairs_keep_move:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} ({sym[move_idx]}) -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg = sum(sym_f == 'Mg')
zn = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn*4 == mg*7 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Moved 198 (Zn) -> inv(202): (6.472, 12.185, 18.193) -> (20.998, 2.558, 7.895)
Moved 144 (Mg) -> inv(142): (13.721, 13.366, 18.228) -> (13.748, 1.377, 7.860)
Moved 154 (Zn) -> inv(152): (5.953, 7.601, 18.269) -> (21.516, 7.141, 7.819)
Moved 150 (Zn) -> inv(151): (26.120, 7.601, 18.269) -> (28.238, 7.141, 7.819)
Moved 145 (Mg) -> inv(143): (11.630, 1.837, 18.311) -> (15.839, 12.906, 7.777)
Moved 196 (Zn) -> inv(200): (18.879, 3.017, 18.346) -> (8.590, 11.725, 7.742)

Atoms: 220, Mg: 80, Zn: 140
Stoichiometric: YES
Symmetric: True


In [11]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [12]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_110_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_110_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg4zn7_110_2layers_reconstructed_ase.data
  Atoms: 220
  Thickness: 10.6 A
  Area: 379.7 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [13]:
#unrelaxed surface energy calculation
E_slab = -249.65418     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 220 / 11                 # formula units in slab = 32
A =    379.7          # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -283.45584 eV
E_slab - n*E_bulk = 33.80166 eV
S = 0.044511 eV/Ang^2
S = 0.7131 J/m^2


In [48]:
#relaxed surface energy calculation
E_slab_relaxed =  -252.61790871325  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 220 / 11                      # 32 formula units
A = 379.7               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7131
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 30.83793 eV
S relaxed = 0.040608 eV/Ang^2
S relaxed = 0.6506 J/m^2

Unrelaxed S = 0.7131 J/m^2
Relaxed S   = 0.6506 J/m^2
Relaxation energy = 0.0625 J/m^2


In [15]:
slab4 = surface(alloy, (1, 1, 0), 4, vacuum=8)

sym4 = np.array(slab4.get_chemical_symbols())
mg4, zn4 = sum(sym4 == 'Mg'), sum(sym4 == 'Zn')
z4 = slab4.get_positions()[:, 2]
thick4 = z4.max() - z4.min()

ps4 = AseAtomsAdaptor().get_structure(slab4)
slab_obj4 = Slab(ps4.lattice, ps4.species, ps4.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps4, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab4)}, Mg: {mg4}, Zn: {zn4}")
print(f"Thickness: {thick4:.1f} A")
print(f"Stoichiometric: {'YES' if zn4*4 == mg4*7 else 'NO'}")
print(f"Symmetric: {slab_obj4.is_symmetric()}")

Atoms: 440, Mg: 160, Zn: 280
Thickness: 20.8 A
Stoichiometric: YES
Symmetric: False


In [16]:
z = slab4.get_positions()[:, 2]
sym = np.array(slab4.get_chemical_symbols())
order = np.argsort(z)

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
print(f"Total atomic layers: {n}\n")

# Search for removals
print("Searching for removals that give Symmetric=True...\n")
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        
        tr = slab4[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            is_sym = slab_tr.is_symmetric()
        except:
            is_sym = False
        
        if is_sym:
            sym_tr = np.array(tr.get_chemical_symbols())
            mg_tr = sum(sym_tr == 'Mg')
            zn_tr = sum(sym_tr == 'Zn')
            removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
            removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
            print(f"  bot_rm={bot_rm}, top_rm={top_rm}: "
                  f"{len(tr)} atoms, Mg{mg_tr} Zn{zn_tr}, "
                  f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")

Total atomic layers: 184

Searching for removals that give Symmetric=True...

  bot_rm=0, top_rm=5: 428 atoms, Mg156 Zn272, removed 0+12=12
  bot_rm=1, top_rm=6: 424 atoms, Mg152 Zn272, removed 2+14=16
  bot_rm=2, top_rm=7: 420 atoms, Mg152 Zn268, removed 4+16=20


In [17]:
# Remove top 5 layers
removed_top = []
for j in range(n - 5, n):
    removed_top.extend(layer_idx[j])

keep = []
for j in range(0, n - 5):
    keep.extend(layer_idx[j])

trimmed = slab4[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

print(f"\nRemoved ({len(removed_top)} atoms):")
rem_mg = sum(sym[j] == 'Mg' for j in removed_top)
rem_zn = sum(sym[j] == 'Zn' for j in removed_top)
print(f"Mg{rem_mg} Zn{rem_zn}")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab4)
keep_set = set(keep)
unpaired = []
for j in removed_top:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    dists = np.linalg.norm(slab4.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    if min_d > 0.5:
        unpaired.append(j)

print(f"Unpaired: {len(unpaired)} (all should be 12)")

SG = P-1
Inversion: f -> [0.3200883  0.07643979 0.99299299] - f

Removed (12 atoms):
Mg4 Zn8
Unpaired: 12 (all should be 12)


In [18]:
# Show removed atoms to identify pairs
for j in removed_top:
    pos = slab4.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Group into pairs by matching y,z (offset in x by ~13.4)
print("\nPairing by x-offset...")
used = set()
pairs_keep_move = []

for i, j1 in enumerate(removed_top):
    if j1 in used:
        continue
    pos1 = slab4.positions[j1]
    for j2 in removed_top[i+1:]:
        if j2 in used:
            continue
        pos2 = slab4.positions[j2]
        # Same element, same y and z, different x
        if sym[j1] == sym[j2] and abs(pos1[1] - pos2[1]) < 0.1 and abs(pos1[2] - pos2[2]) < 0.1:
            pairs_keep_move.append((j1, j2))
            used.add(j1)
            used.add(j2)
            print(f"  Pair: ({j1}, {j2}) {sym[j1]}")
            break

# Reconstruct: keep first, move second to inv(first)
sc_fixed4 = slab4.copy()
for keep_idx, move_idx in pairs_keep_move:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed4.positions[move_idx] = inv_cart
    print(f"  Moved {move_idx} -> inv({keep_idx})")

sym_f = np.array(sc_fixed4.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed4 = AseAtomsAdaptor().get_structure(sc_fixed4)
slab_fixed4 = Slab(ps_fixed4.lattice, ps_fixed4.species, ps_fixed4.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed4, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed4)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
print(f"Symmetric: {slab_fixed4.is_symmetric()}")

  idx=422 Zn (17.798, 12.645, 28.644)
  idx=418 Zn (4.353, 12.645, 28.644)
  idx=362 Mg (25.047, 13.825, 28.678)
  idx=364 Mg (11.603, 13.825, 28.678)
  idx=370 Zn (24.001, 8.061, 28.720)
  idx=374 Zn (3.835, 8.061, 28.720)
  idx=372 Zn (17.279, 8.061, 28.720)
  idx=371 Zn (10.557, 8.061, 28.720)
  idx=365 Mg (9.512, 2.297, 28.762)
  idx=363 Mg (22.956, 2.297, 28.762)
  idx=420 Zn (3.317, 3.477, 28.797)
  idx=416 Zn (16.761, 3.477, 28.797)

Pairing by x-offset...
  Pair: (422, 418) Zn
  Pair: (362, 364) Mg
  Pair: (370, 374) Zn
  Pair: (372, 371) Zn
  Pair: (365, 363) Mg
  Pair: (420, 416) Zn
  Moved 418 -> inv(422)
  Moved 364 -> inv(362)
  Moved 374 -> inv(370)
  Moved 371 -> inv(372)
  Moved 363 -> inv(365)
  Moved 416 -> inv(420)

Atoms: 440, Mg: 160, Zn: 280
Stoichiometric: YES
Symmetric: True


In [19]:
view(sc_fixed4)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [20]:
thick4 = sc_fixed4.get_positions()[:,2].max() - sc_fixed4.get_positions()[:,2].min()
area4 = np.linalg.norm(np.cross(sc_fixed4.cell[0], sc_fixed4.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed4)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_110_4layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_110_4layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed4)}")
print(f"  Thickness: {thick4:.1f} A")
print(f"  Area: {area4:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg4zn7_110_4layers_reconstructed_ase.data
  Atoms: 440
  Thickness: 21.1 A
  Area: 379.7 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [21]:
#unrelaxed surface energy calculation
E_slab =  -531.29123     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 440 / 11                 # formula units in slab = 32
A =    379.7          # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -566.91168 eV
E_slab - n*E_bulk = 35.62045 eV
S = 0.046906 eV/Ang^2
S = 0.7515 J/m^2


In [47]:
#relaxed surface energy calculation
E_slab_relaxed = -536.076092855852  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 440 / 11                      # 32 formula units
A = 379.7               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7515
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 30.83559 eV
S relaxed = 0.040605 eV/Ang^2
S relaxed = 0.6506 J/m^2

Unrelaxed S = 0.7515 J/m^2
Relaxed S   = 0.6506 J/m^2
Relaxation energy = 0.1009 J/m^2


In [23]:
slab6 = surface(alloy, (1, 1, 0), 6, vacuum=8)

sym6 = np.array(slab6.get_chemical_symbols())
mg6, zn6 = sum(sym6 == 'Mg'), sum(sym6 == 'Zn')
z6 = slab6.get_positions()[:, 2]
thick6 = z6.max() - z6.min()

print(f"Atoms: {len(slab6)}, Mg: {mg6}, Zn: {zn6}")
print(f"Thickness: {thick6:.1f} A")
print(f"Stoichiometric: {'YES' if zn6*4 == mg6*7 else 'NO'}")

Atoms: 660, Mg: 240, Zn: 420
Thickness: 31.2 A
Stoichiometric: YES


In [24]:
z = slab6.get_positions()[:, 2]
sym = np.array(slab6.get_chemical_symbols())
order = np.argsort(z)

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
print(f"Atomic layers: {n}")

# Find symmetric removal
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab6[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                sym_tr = np.array(tr.get_chemical_symbols())
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                break
        except:
            continue
    else:
        continue
    break

# Get inversion
sga = SpacegroupAnalyzer(ps_tr, symprec=0.1)
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Get removed atoms and pair them
removed_top_idx = []
for j in range(n - top_rm, n):
    removed_top_idx.extend(layer_idx[j])

print(f"\nRemoved: {len(removed_top_idx)} atoms")

# Auto-pair by matching y,z
ps_orig = AseAtomsAdaptor().get_structure(slab6)
used = set()
pairs = []
for i, j1 in enumerate(removed_top_idx):
    if j1 in used:
        continue
    pos1 = slab6.positions[j1]
    for j2 in removed_top_idx[i+1:]:
        if j2 in used:
            continue
        pos2 = slab6.positions[j2]
        if sym[j1] == sym[j2] and abs(pos1[1]-pos2[1]) < 0.1 and abs(pos1[2]-pos2[2]) < 0.1:
            pairs.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

print(f"Pairs found: {len(pairs)}")

# Reconstruct
sc_fixed6 = slab6.copy()
for keep_idx, move_idx in pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed6.positions[move_idx] = inv_cart

sym_f = np.array(sc_fixed6.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed6 = AseAtomsAdaptor().get_structure(sc_fixed6)
slab_fixed6 = Slab(ps_fixed6.lattice, ps_fixed6.species, ps_fixed6.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed6, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed6)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
print(f"Symmetric: {slab_fixed6.is_symmetric()}")

Atomic layers: 276
  bot_rm=0, top_rm=5: 648 atoms, removed 0+12=12
Inversion: f -> [0.23759398 0.10899654 0.99454297] - f

Removed: 12 atoms
Pairs found: 6

Atoms: 660, Mg: 240, Zn: 420
Stoichiometric: YES
Symmetric: True


In [25]:
view(sc_fixed6)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [26]:
thick6 = sc_fixed6.get_positions()[:,2].max() - sc_fixed6.get_positions()[:,2].min()
area6 = np.linalg.norm(np.cross(sc_fixed6.cell[0], sc_fixed6.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed6)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_110_6layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_110_6layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed6)}")
print(f"  Thickness: {thick6:.1f} A")
print(f"  Area: {area6:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg4zn7_110_6layers_reconstructed_ase.data
  Atoms: 660
  Thickness: 31.5 A
  Area: 379.7 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [27]:
#unrelaxed surface energy calculation
E_slab =  -812.83322     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 660 / 11                 # formula units in slab = 32
A =    379.7          # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -850.36752 eV
E_slab - n*E_bulk = 37.53430 eV
S = 0.049426 eV/Ang^2
S = 0.7919 J/m^2


In [46]:
#relaxed surface energy calculation
E_slab_relaxed = -819.009437475198  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 660 / 11                      # 32 formula units
A = 379.7               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7919
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 31.35808 eV
S relaxed = 0.041293 eV/Ang^2
S relaxed = 0.6616 J/m^2

Unrelaxed S = 0.7919 J/m^2
Relaxed S   = 0.6616 J/m^2
Relaxation energy = 0.1303 J/m^2


In [29]:
slab8 = surface(alloy, (1, 1, 0), 8, vacuum=8)
z = slab8.get_positions()[:, 2]
sym = np.array(slab8.get_chemical_symbols())
order = np.argsort(z)
thick8_raw = z.max() - z.min()
print(f"Atoms: {len(slab8)}, Thickness: {thick8_raw:.1f} A")

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
print(f"Atomic layers: {n}")

# Find symmetric removal
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab8[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"removed {removed_bot}+{removed_top}")
                break
        except:
            continue
    else:
        continue
    break

# Get inversion
sga = SpacegroupAnalyzer(ps_tr, symprec=0.1)
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Get removed atoms and auto-pair
removed_top_idx = []
for j in range(n - top_rm, n):
    removed_top_idx.extend(layer_idx[j])

ps_orig = AseAtomsAdaptor().get_structure(slab8)
used = set()
pairs = []
for i, j1 in enumerate(removed_top_idx):
    if j1 in used:
        continue
    pos1 = slab8.positions[j1]
    for j2 in removed_top_idx[i+1:]:
        if j2 in used:
            continue
        pos2 = slab8.positions[j2]
        if sym[j1] == sym[j2] and abs(pos1[1]-pos2[1]) < 0.1 and abs(pos1[2]-pos2[2]) < 0.1:
            pairs.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

# Reconstruct
sc_fixed8 = slab8.copy()
for keep_idx, move_idx in pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed8.positions[move_idx] = inv_cart

sym_f = np.array(sc_fixed8.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed8 = AseAtomsAdaptor().get_structure(sc_fixed8)
slab_fixed8 = Slab(ps_fixed8.lattice, ps_fixed8.species, ps_fixed8.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed8, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick8 = sc_fixed8.get_positions()[:,2].max() - sc_fixed8.get_positions()[:,2].min()
area8 = np.linalg.norm(np.cross(sc_fixed8.cell[0], sc_fixed8.cell[1]))

print(f"\nAtoms: {len(sc_fixed8)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
print(f"Symmetric: {slab_fixed8.is_symmetric()}")
print(f"Thickness: {thick8:.1f} A")
print(f"Area: {area8:.1f} A²")

Atoms: 880, Thickness: 41.7 A
Atomic layers: 368
  bot_rm=0, top_rm=5: 868 atoms, removed 0+12
Inversion: f -> [0.15510204 0.14155713 0.99553073] - f

Atoms: 880, Mg: 320, Zn: 560
Stoichiometric: YES
Symmetric: True
Thickness: 42.0 A
Area: 379.7 A²


In [30]:
view(sc_fixed8)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [31]:
ps = AseAtomsAdaptor().get_structure(sc_fixed8)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_110_8layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_110_8layers_reconstructed_ase.data")

Saved: slabs/mg4zn7_110_8layers_reconstructed_ase.data


In [33]:
#unrelaxed surface energy calculation
E_slab =  -1094.4174     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 880 / 11                 # formula units in slab = 32
A =    379.7          # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -1133.82336 eV
E_slab - n*E_bulk = 39.40596 eV
S = 0.051891 eV/Ang^2
S = 0.8314 J/m^2


In [45]:
#relaxed surface energy calculation
E_slab_relaxed = -1102.2680352464  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 880 / 11                      # 32 formula units
A = 379.7               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.8314
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 31.55532 eV
S relaxed = 0.041553 eV/Ang^2
S relaxed = 0.6658 J/m^2

Unrelaxed S = 0.8314 J/m^2
Relaxed S   = 0.6658 J/m^2
Relaxation energy = 0.1656 J/m^2


In [35]:
slab10 = surface(alloy, (1, 1, 0), 10, vacuum=8)
z = slab10.get_positions()[:, 2]
sym = np.array(slab10.get_chemical_symbols())
order = np.argsort(z)
print(f"Atoms: {len(slab10)}, Thickness: {z.max()-z.min():.1f} A")

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
print(f"Atomic layers: {n}")

for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab10[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"removed {removed_bot}+{removed_top}")
                break
        except:
            continue
    else:
        continue
    break

sga = SpacegroupAnalyzer(ps_tr, symprec=0.1)
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

removed_top_idx = []
for j in range(n - top_rm, n):
    removed_top_idx.extend(layer_idx[j])

ps_orig = AseAtomsAdaptor().get_structure(slab10)
used = set()
pairs = []
for i, j1 in enumerate(removed_top_idx):
    if j1 in used:
        continue
    pos1 = slab10.positions[j1]
    for j2 in removed_top_idx[i+1:]:
        if j2 in used:
            continue
        pos2 = slab10.positions[j2]
        if sym[j1] == sym[j2] and abs(pos1[1]-pos2[1]) < 0.1 and abs(pos1[2]-pos2[2]) < 0.1:
            pairs.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

sc_fixed10 = slab10.copy()
for keep_idx, move_idx in pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed10.positions[move_idx] = inv_cart

sym_f = np.array(sc_fixed10.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed10 = AseAtomsAdaptor().get_structure(sc_fixed10)
slab_fixed10 = Slab(ps_fixed10.lattice, ps_fixed10.species, ps_fixed10.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed10, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick10 = sc_fixed10.get_positions()[:,2].max() - sc_fixed10.get_positions()[:,2].min()
area10 = np.linalg.norm(np.cross(sc_fixed10.cell[0], sc_fixed10.cell[1]))

print(f"\nAtoms: {len(sc_fixed10)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
print(f"Symmetric: {slab_fixed10.is_symmetric()}")
print(f"Thickness: {thick10:.1f} A")
print(f"Area: {area10:.1f} A²")

Atoms: 1100, Thickness: 52.1 A
Atomic layers: 460
  bot_rm=0, top_rm=5: 1088 atoms, removed 0+12
Inversion: f -> [0.07260726 0.17411402 0.9962169 ] - f

Atoms: 1100, Mg: 400, Zn: 700
Stoichiometric: YES
Symmetric: True
Thickness: 52.4 A
Area: 379.7 A²


In [36]:
ps = AseAtomsAdaptor().get_structure(sc_fixed10)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_110_10layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_110_10layers_reconstructed_ase.data")

Saved: slabs/mg4zn7_110_10layers_reconstructed_ase.data


In [37]:
#unrelaxed surface energy calculation
E_slab =  -1376.1769     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 1100 / 11                 # formula units in slab = 32
A =    379.7          # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -1417.27920 eV
E_slab - n*E_bulk = 41.10230 eV
S = 0.054125 eV/Ang^2
S = 0.8672 J/m^2


In [44]:
#relaxed surface energy calculation
E_slab_relaxed =  -1385.02765458446  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 1100 / 11                      # formula units
A = 379.7               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.8672
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 32.25155 eV
S relaxed = 0.042470 eV/Ang^2
S relaxed = 0.6804 J/m^2

Unrelaxed S = 0.8672 J/m^2
Relaxed S   = 0.6804 J/m^2
Relaxation energy = 0.1868 J/m^2


In [41]:
# Check the 10-layer slab
pos10 = sc_fixed10.get_positions()
sym10 = np.array(sc_fixed10.get_chemical_symbols())

# Find the moved atoms (they're at z < 9 or so — near bottom)
z10 = pos10[:, 2]
bottom_atoms = np.where(z10 < z10.min() + 1.0)[0]

print("Bottom surface atoms (including reconstructed):")
for i in bottom_atoms:
    # Find nearest neighbour distance
    dists = np.linalg.norm(pos10 - pos10[i], axis=1)
    dists[i] = 999  # exclude self
    nn_dist = dists.min()
    nn_idx = dists.argmin()
    print(f"  idx={i} {sym10[i]} z={z10[i]:.3f} | nearest: idx={nn_idx} {sym10[nn_idx]} dist={nn_dist:.3f} A")

Bottom surface atoms (including reconstructed):
  idx=9 Mg z=8.658 | nearest: idx=55 Zn dist=3.104 A
  idx=11 Mg z=8.658 | nearest: idx=57 Zn dist=3.104 A
  idx=17 Mg z=8.454 | nearest: idx=81 Zn dist=3.117 A
  idx=19 Mg z=8.454 | nearest: idx=85 Zn dist=3.117 A
  idx=21 Mg z=8.000 | nearest: idx=105 Zn dist=3.053 A
  idx=23 Mg z=8.000 | nearest: idx=1024 Mg dist=3.031 A
  idx=47 Zn z=8.227 | nearest: idx=81 Zn dist=2.639 A
  idx=49 Zn z=8.227 | nearest: idx=85 Zn dist=2.639 A
  idx=51 Zn z=8.699 | nearest: idx=105 Zn dist=2.666 A
  idx=53 Zn z=8.699 | nearest: idx=109 Zn dist=2.666 A
  idx=94 Zn z=8.533 | nearest: idx=61 Zn dist=2.686 A
  idx=98 Zn z=8.533 | nearest: idx=59 Zn dist=2.686 A
  idx=102 Zn z=8.686 | nearest: idx=103 Zn dist=2.629 A
  idx=106 Zn z=8.686 | nearest: idx=107 Zn dist=2.629 A
  idx=1024 Mg z=7.861 | nearest: idx=101 Zn dist=2.953 A
  idx=1025 Mg z=7.777 | nearest: idx=98 Zn dist=2.953 A
  idx=1032 Zn z=7.819 | nearest: idx=102 Zn dist=2.685 A
  idx=1034 Zn z=7.

In [42]:
for label, n_layers, E_slab_unrelaxed, E_slab_relaxed in [
    ("2L", 2, None, None),  # fill in your values
    ("4L", 4, None, None),
    ("6L", 6, None, None),
    ("8L", 8, None, None),
    ("10L", 10, None, None),
]:
    slab_test = surface(alloy, (1, 1, 0), n_layers, vacuum=8)
    sym_t = np.array(slab_test.get_chemical_symbols())
    mg_t = sum(sym_t == 'Mg')
    zn_t = sum(sym_t == 'Zn')
    n_fu = mg_t // 4
    area = np.linalg.norm(np.cross(slab_test.cell[0], slab_test.cell[1]))
    z_t = slab_test.get_positions()[:, 2]
    thick = z_t.max() - z_t.min()
    
    print(f"{label}: {len(slab_test)} atoms, Mg{mg_t} Zn{zn_t}, "
          f"n_fu={n_fu}, area={area:.1f} A², thick={thick:.1f} A")

2L: 220 atoms, Mg80 Zn140, n_fu=20, area=379.7 A², thick=10.3 A
4L: 440 atoms, Mg160 Zn280, n_fu=40, area=379.7 A², thick=20.8 A
6L: 660 atoms, Mg240 Zn420, n_fu=60, area=379.7 A², thick=31.2 A
8L: 880 atoms, Mg320 Zn560, n_fu=80, area=379.7 A², thick=41.7 A
10L: 1100 atoms, Mg400 Zn700, n_fu=100, area=379.7 A², thick=52.1 A


In [43]:
E_bulk_per_fu = -141.72792 / 10
A = 379.7

print(f"{'Layers':>6} {'E_unrel':>10} {'E_rel':>10} {'n':>4} {'S_unrel':>8} {'S_rel':>8} {'S_relax':>8}")
print("-" * 65)

# Fill in your E_slab values:
data = [
    (2,  -249.65418, -252.617, 20),  
    (4,  -531.29123, -536.076, 40),
    (6,  -812.83322, -819.009, 60),
    (8,  -1094.4174 , -1102.268, 80),
    (10, -1376.1769, -1385.027, 100),
]

for layers, E_unrel, E_rel, n in data:
    if E_unrel is None:
        continue
    nE = n * E_bulk_per_fu
    S_un = (E_unrel - nE) / (2 * A) * 16.0218
    S_re = (E_rel - nE) / (2 * A) * 16.0218
    print(f"{layers:>6} {E_unrel:>10.3f} {E_rel:>10.3f} {n:>4} {S_un:>8.4f} {S_re:>8.4f} {S_un-S_re:>8.4f}")

Layers    E_unrel      E_rel    n  S_unrel    S_rel  S_relax
-----------------------------------------------------------------
     2   -249.654   -252.617   20   0.7131   0.6506   0.0625
     4   -531.291   -536.076   40   0.7515   0.6506   0.1009
     6   -812.833   -819.009   60   0.7919   0.6616   0.1303
     8  -1094.417  -1102.268   80   0.8314   0.6658   0.1656
    10  -1376.177  -1385.027  100   0.8672   0.6805   0.1867
